# DermAI: HAM10000 Exploratory Data Analysis (EDA)
### Milestone 1 — Dataset Selection, Structure, & Pre-Splitting Diagnostics

> **Regulatory & Clinical Notice**: *AI-assisted analysis, not a standalone medical diagnosis.*

This notebook conducts systematic EDA on the 10,015 dermatoscopic images from the **HAM10000** dataset:
- 7 Target Disease Classes: Melanoma (`mel`), Melanocytic Nevus (`nv`), Basal Cell Carcinoma (`bcc`), Actinic Keratoses (`akiec`), Benign Keratosis (`bkl`), Dermatofibroma (`df`), Vascular Lesion (`vasc`)
- Patient Demographics (Age, Sex, Anatomical Localization)
- Lesion Duplication (`lesion_id` grouped distribution)
- Image Dimension & Aspect Ratio Statistics
- Class Weight Calculation for Imbalance Mitigation

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

metadata_path = Path('data/raw/ham10000/HAM10000_metadata.csv')
print(f"Checking metadata path: {metadata_path.resolve()}")
assert metadata_path.exists(), f"Missing {metadata_path}"
df = pd.read_csv(metadata_path)
print(f"Loaded {len(df):,} image metadata rows.")
df.head()

## 1. Class Distribution Analysis
Examination of the 7 diagnostic categories (`dx`), mapping abbreviations to clinical diagnoses.

In [2]:
DIAGNOSIS_MAP = {
    'nv': 'Melanocytic Nevus (Benign)',
    'mel': 'Melanoma (Malignant)',
    'bkl': 'Benign Keratosis-like Lesion',
    'bcc': 'Basal Cell Carcinoma (Malignant)',
    'akiec': 'Actinic Keratosis / Bowen\'s Disease',
    'vasc': 'Vascular Lesion (Benign)',
    'df': 'Dermatofibroma (Benign)'
}

class_counts = df['dx'].value_counts()
class_pcts = df['dx'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Code': class_counts.index,
    'Diagnosis': [DIAGNOSIS_MAP.get(c, c) for c in class_counts.index],
    'Count': class_counts.values,
    'Percentage (%)': class_pcts.round(2).values
})
print(dist_df.to_string(index=False))

print("\n--- ASCII Distribution Bar Chart ---")
for _, row in dist_df.iterrows():
    bar = '#' * int(row['Percentage (%)'] / 2)
    print(f"{row['Code']:<6} | {bar:<35} {row['Percentage (%)']:>5.1f}% ({row['Count']} imgs)")

## 2. Demographic Analysis: Age and Sex Distribution

In [3]:
print("=== Sex Distribution ===")
sex_counts = df['sex'].value_counts(dropna=False)
print(sex_counts)

print("\n=== Age Statistics ===")
age_stats = df['age'].describe()
print(age_stats)
print(f"Missing age values: {df['age'].isna().sum()} records ({df['age'].isna().mean()*100:.2f}%)")

## 3. Lesion Localization (Body Site Distribution)

In [4]:
loc_counts = df['localization'].value_counts(dropna=False)
loc_pcts = df['localization'].value_counts(normalize=True) * 100
loc_df = pd.DataFrame({
    'Anatomical Site': loc_counts.index,
    'Image Count': loc_counts.values,
    'Percentage (%)': loc_pcts.round(2).values
})
print(loc_df.to_string(index=False))

## 4. Multi-Image Lesion Inspection & Data Leakage Prevention
Checking how many images belong to the same `lesion_id`. Random splitting without grouping by `lesion_id` leaks identical patient lesions between train and test!

In [5]:
images_per_lesion = df.groupby('lesion_id').size()
print(f"Total unique lesions: {len(images_per_lesion):,}")
print(f"Total images: {len(df):,}")
print(f"Max images for a single lesion: {images_per_lesion.max()}")
print(f"Mean images per lesion: {images_per_lesion.mean():.3f}")
print("\nDistribution of images per lesion:")
print(images_per_lesion.value_counts().sort_index())

## 5. Image Size Statistics from Sample Directory

In [6]:
samples_dir = Path('data/raw/ham10000/sample_images')
if samples_dir.exists():
    sample_files = list(samples_dir.glob('*.jpg'))
    print(f"Found {len(sample_files)} sample benchmark images.")
    widths, heights, aspect_ratios, file_sizes_kb = [], [], [], []
    for p in sample_files:
        with Image.open(p) as im:
            w, h = im.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(round(w / h, 3))
            file_sizes_kb.append(round(p.stat().st_size / 1024, 1))
            
    size_df = pd.DataFrame({
        'Width (px)': widths,
        'Height (px)': heights,
        'Aspect Ratio (W/H)': aspect_ratios,
        'File Size (KB)': file_sizes_kb
    })
    print(size_df.describe().round(2))
else:
    print("Standard HAM10000 dimension: 600 x 450 pixels (4:3 aspect ratio), 24-bit RGB.")

## 6. Three Sample Benchmark Images Per Class

In [7]:
classes = ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
samples_table = []
for c in classes:
    files = sorted(list(samples_dir.glob(f'{c}_sample_*.jpg')))
    samples_table.append({
        'Class': c,
        'Diagnosis': DIAGNOSIS_MAP[c],
        'Available Samples': [f.name for f in files]
    })
samples_summary_df = pd.DataFrame(samples_table)
print(samples_summary_df.to_string(index=False))

## 7. Computation of Balanced Class Weights

In [8]:
n_total = len(df)
n_classes = len(class_counts)
class_weights = {
    cls: round(float(n_total / (n_classes * count)), 4)
    for cls, count in class_counts.items()
}
print("Balanced Class Weights (Entire Dataset):")
for cls, w in sorted(class_weights.items(), key=lambda x: x[1]):
    print(f"  {cls:<6} ({DIAGNOSIS_MAP[cls]:<35}): weight = {w:8.4f}")